In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [2]:
score_data = pd.read_csv('scores.csv', index_col=[0,1])
score_data

,,like_per_month,newfriend_per_month,post_per_month,adview_per_month,dislike_per_month,unfriend_per_month,message_per_month,reply_per_month,account_tenure,adview_per_post,...,like_per_post,post_per_message,unfriend_per_newfriend,dislike_pcnt,unfriend_per_newfriend_scaled,newfriend_pcnt_chng,days_since_newfriend,unfriend_28day_avg_84day_obs,unfriend_28day_avg_84day_obs_scaled,is_churn
account_id,observation_date,,,,,,,,,,,,,,,,,,,,,
27,2020-03-01,-0.252128,0.176105,-0.250193,0.449681,-1.074832,1.271113,-1.913326,-1.681545,-0.600623,0.794364,...,-0.151952,2.123692,0.702322,-0.789858,0.165914,3.220524,-0.073812,0.310313,0.505493,False
51,2020-03-01,-0.306824,0.176105,-1.186311,0.010731,-0.593460,1.271113,0.397880,0.851668,-0.600623,1.777259,...,1.111640,-0.860022,0.702322,-0.468472,0.804718,-0.436188,-0.683207,1.444420,1.792298,False
95,2020-03-01,0.475167,0.176105,0.999259,-0.076426,0.758881,-0.551847,-0.503272,-0.217639,-0.600623,-0.987500,...,-0.607365,1.140677,-0.403228,-0.085544,-0.534873,0.232470,-0.683207,-0.823795,-0.781313,False
123,2020-03-01,-0.252128,0.347369,-0.308981,-1.136047,0.960141,-0.551847,-0.563570,-1.160091,-0.600623,-0.927775,...,-0.082986,-0.074285,-0.403228,1.162712,-0.534873,1.257860,-0.326734,-0.823795,-0.781313,False
189,2020-03-01,-1.768424,-1.044471,-0.685463,-0.123708,-1.074832,-0.551847,-2.435484,-1.681545,-0.600623,0.588915,...,-1.227813,-0.963738,-0.403228,0.817071,-0.534873,-0.167825,1.144979,-0.823795,-0.781313,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13132,2020-05-10,-0.536530,-1.814568,0.036512,-1.558615,-0.803737,-0.551847,0.074712,-0.217639,-1.722229,-1.243688,...,-0.746014,-0.331772,-0.403228,-0.453864,-0.534873,-0.167825,-1.292602,-0.823795,-0.781313,False
13141,2020-05-10,-1.768424,-1.044471,-0.899606,-0.562658,-1.456919,-0.551847,-0.447446,-0.333606,-1.722229,0.255892,...,-1.107822,-0.531085,-0.403228,0.093953,-0.534873,-0.167825,1.296065,-0.823795,-0.781313,False
13155,2020-05-10,-1.988622,-1.814568,-1.967073,-1.894027,-1.456919,-0.551847,0.074712,0.492879,-1.722229,-0.151696,...,-0.151952,-0.922276,-0.403228,0.495685,-0.534873,-0.167825,-1.292602,-0.823795,-0.781313,False


In [3]:
score_data.drop('is_churn',axis=1, inplace=True)

In [4]:
#estas son antes de ordenarlas
metric_columns = list(score_data.columns.values)
metric_columns

['like_per_month',
 'newfriend_per_month',
 'post_per_month',
 'adview_per_month',
 'dislike_per_month',
 'unfriend_per_month',
 'message_per_month',
 'reply_per_month',
 'account_tenure',
 'adview_per_post',
 'reply_per_message',
 'like_per_post',
 'post_per_message',
 'unfriend_per_newfriend',
 'dislike_pcnt',
 'unfriend_per_newfriend_scaled',
 'newfriend_pcnt_chng',
 'days_since_newfriend',
 'unfriend_28day_avg_84day_obs',
 'unfriend_28day_avg_84day_obs_scaled']

In [5]:
group_corr_thresh = 0.5
corr = score_data.corr()

dissimilarity = 1.0 - corr
diss_thresh = 1.0 - group_corr_thresh

#entre mas chiquito menos distance matrix redundant
dissimilarity

,like_per_month,newfriend_per_month,post_per_month,adview_per_month,dislike_per_month,unfriend_per_month,message_per_month,reply_per_month,account_tenure,adview_per_post,reply_per_message,like_per_post,post_per_message,unfriend_per_newfriend,dislike_pcnt,unfriend_per_newfriend_scaled,newfriend_pcnt_chng,days_since_newfriend,unfriend_28day_avg_84day_obs,unfriend_28day_avg_84day_obs_scaled
like_per_month,0.000000,0.390769,0.336151,0.329404,0.546429,1.005232,0.889335,0.929611,0.951035,0.911668,1.049686,0.497125,0.639912,1.104178,1.551707,1.150400,0.926857,1.312686,0.991750,1.010971
newfriend_per_month,0.390769,0.000000,0.406156,0.399087,0.588708,1.002017,0.907408,0.943465,0.946180,0.923883,1.038376,0.887218,0.669955,1.152618,1.223873,1.219113,0.644100,1.507055,0.983953,1.007108
post_per_month,0.336151,0.406156,0.000000,0.337115,0.568047,1.000456,0.894865,0.932718,0.948230,1.261298,1.041615,1.219253,0.430080,1.095937,1.254245,1.140475,0.925043,1.301684,0.983097,1.003354
adview_per_month,0.329404,0.399087,0.337115,0.000000,0.550900,1.000579,0.866415,0.907785,0.956494,0.546845,1.040500,0.884035,0.657759,1.096327,1.247728,1.140210,0.928032,1.308008,0.984817,1.003927
dislike_per_month,0.546429,0.588708,0.568047,0.550900,0.000000,0.991811,0.848964,0.881759,0.960371,0.936250,1.020213,0.912100,0.819839,1.072143,0.599236,1.104740,0.946351,1.216701,0.987647,1.006124
unfriend_per_month,1.005232,1.002017,1.000456,1.000579,0.991811,0.000000,1.002121,0.998721,0.989331,0.998563,0.988288,1.007916,0.997982,0.297300,0.988800,0.563720,0.995930,0.999205,0.370316,0.321910
message_per_month,0.889335,0.907408,0.894865,0.866415,0.848964,1.002121,0.000000,0.118187,0.959851,0.963965,0.965976,0.978517,1.558576,1.009009,0.982584,1.013088,0.994465,1.039760,0.978714,0.994068
reply_per_month,0.929611,0.943465,0.932718,0.907785,0.881759,0.998721,0.118187,0.000000,0.965640,0.972264,0.584466,0.986932,1.520405,1.002841,0.973043,1.004298,0.997902,1.019189,0.979127,0.993248
account_tenure,0.951035,0.946180,0.948230,0.956494,0.960371,0.989331,0.959851,0.965640,0.000000,1.001331,0.994180,0.994673,0.989465,1.000199,1.010165,0.995931,1.022780,0.917648,0.769896,1.007444
adview_per_post,0.911668,0.923883,1.261298,0.546845,0.936250,0.998563,0.963965,0.972264,1.001331,0.000000,1.004941,0.489743,1.197288,1.001198,1.033004,1.006924,0.985197,1.025722,0.997477,1.000183


In [6]:
#1D condensed distance matrix
linkage_matrix = linkage(squareform(dissimilarity), method='single')

In [7]:
labels = fcluster(
    linkage_matrix,diss_thresh,criterion='distance')

labels

array([6, 6, 6, 6, 7, 1, 4, 4, 2, 6, 5, 6, 6, 1, 8, 1, 9, 3, 1, 1],
      dtype=int32)

In [8]:
#counts the number of elements on each cluster
cluster_count = Counter(labels)

#este diccionarion nadamas tiene los indices
cluster_order = {cluster[0] : idx for idx, cluster in enumerate(cluster_count.most_common())}

cluster_order

{np.int32(6): 0,
 np.int32(1): 1,
 np.int32(4): 2,
 np.int32(7): 3,
 np.int32(2): 4,
 np.int32(5): 5,
 np.int32(8): 6,
 np.int32(9): 7,
 np.int32(3): 8}

In [9]:
relabeled_clusters = [cluster_order[l] for l in labels]

In [10]:
relabeled_count = Counter(relabeled_clusters)
relabeled_count

Counter({0: 7, 1: 5, 2: 2, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1})

In [11]:
relabeled_df = pd.DataFrame({
    'group' : relabeled_clusters,
    'column' : metric_columns
}).sort_values(['group','column'], ascending=[True,True])
relabeled_df

,group,column
3,0,adview_per_month
9,0,adview_per_post
0,0,like_per_month
11,0,like_per_post
1,0,newfriend_per_month
12,0,post_per_message
2,0,post_per_month
18,1,unfriend_28day_avg_84day_obs
19,1,unfriend_28day_avg_84day_obs_scaled
5,1,unfriend_per_month


In [12]:
#originals X clusters
load_mat = np.zeros((len(metric_columns), len(relabeled_count)))
load_mat

array([[0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [13]:
for row in relabeled_df.iterrows():
    #estas accediendo a la serie
    #print(row[1]['group']) - 0
    #print(row[1]['column']) - 1
    
    orig_col = metric_columns.index(row[1]['column'])
    if relabeled_count[row[1]['group']] > 1:
        load_mat[orig_col, row[1]['group']] = (
            1.0 / 
            (
                np.sqrt(group_corr_thresh) * 
                float(relabeled_count[row[1]['group']])
            )                                
        )
    else:
        load_mat[orig_col, row[1]['group']] = 1.0

In [14]:
is_group = load_mat.astype(bool).sum(axis=0) > 1
is_group

array([ True,  True,  True, False, False, False, False, False, False])

In [15]:
column_names = [
    'metric_group_{}'.format(d) 
    if is_group[d] 
    else 
    relabeled_df.loc[relabeled_df['group'] == d, 'column'].item()
    for d in range(0,load_mat.shape[1])
]
column_names    

['metric_group_0',
 'metric_group_1',
 'metric_group_2',
 'dislike_per_month',
 'account_tenure',
 'reply_per_message',
 'dislike_pcnt',
 'newfriend_pcnt_chng',
 'days_since_newfriend']

In [16]:
loadmat_df = pd.DataFrame(load_mat, index=metric_columns, columns=column_names)
loadmat_df

,metric_group_0,metric_group_1,metric_group_2,dislike_per_month,account_tenure,reply_per_message,dislike_pcnt,newfriend_pcnt_chng,days_since_newfriend
like_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
newfriend_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
post_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
adview_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
dislike_per_month,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0
unfriend_per_month,0.000000,0.282843,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
message_per_month,0.000000,0.000000,0.707107,0.0,0.0,0.0,0.0,0.0,0.0
reply_per_month,0.000000,0.000000,0.707107,0.0,0.0,0.0,0.0,0.0,0.0
account_tenure,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0
adview_per_post,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
loadmat_df.to_csv('load_matrix.csv')

In [18]:
loadmat_df

,metric_group_0,metric_group_1,metric_group_2,dislike_per_month,account_tenure,reply_per_message,dislike_pcnt,newfriend_pcnt_chng,days_since_newfriend
like_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
newfriend_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
post_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
adview_per_month,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
dislike_per_month,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0
unfriend_per_month,0.000000,0.282843,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
message_per_month,0.000000,0.000000,0.707107,0.0,0.0,0.0,0.0,0.0,0.0
reply_per_month,0.000000,0.000000,0.707107,0.0,0.0,0.0,0.0,0.0,0.0
account_tenure,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0
adview_per_post,0.202031,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
group_lists = [
    '|'.join(relabeled_df[relabeled_df['group'] == g]['column']) 
    for g in set(relabeled_df['group'])
]
group_lists

['adview_per_month|adview_per_post|like_per_month|like_per_post|newfriend_per_month|post_per_message|post_per_month',
 'unfriend_28day_avg_84day_obs|unfriend_28day_avg_84day_obs_scaled|unfriend_per_month|unfriend_per_newfriend|unfriend_per_newfriend_scaled',
 'message_per_month|reply_per_month',
 'dislike_per_month',
 'account_tenure',
 'reply_per_message',
 'dislike_pcnt',
 'newfriend_pcnt_chng',
 'days_since_newfriend']

In [24]:
groupmets = pd.DataFrame(group_lists, columns=['metrics'], index=loadmat_df.columns.values)
groupmets

,metrics
metric_group_0,adview_per_month|adview_per_post|like_per_mont...
metric_group_1,unfriend_28day_avg_84day_obs|unfriend_28day_av...
metric_group_2,message_per_month|reply_per_month
dislike_per_month,dislike_per_month
account_tenure,account_tenure
reply_per_message,reply_per_message
dislike_pcnt,dislike_pcnt
newfriend_pcnt_chng,newfriend_pcnt_chng
days_since_newfriend,days_since_newfriend


In [25]:
groupmets.to_csv('groupmets.csv')